In [0]:
base_path = "/Volumes/workspace/employee/employee_assessment_files/"

display(dbutils.fs.ls(base_path))

path,name,size,modificationTime
dbfs:/Volumes/workspace/employee/employee_assessment_files/attendance.csv,attendance.csv,620,1794397101000
dbfs:/Volumes/workspace/employee/employee_assessment_files/tasks.csv,tasks.csv,404,1794397101000


In [0]:
attendance_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(base_path + "attendance.csv")
)

display(attendance_df)

employee_id,employee_name,department,attendance_date,clock_in,clock_out,break_minutes,attendance_status
1,Arun,IT,2026-10-12,09:00:00,17:00:00,60,Present
2,Priya,HR,2026-10-12,09:15:00,17:15:00,45,Present
3,Rahul,Finance,2026-10-12,null,null,0,Absent
4,Meena,IT,2026-10-12,09:05:00,17:05:00,60,Present
1,Arun,IT,2026-10-13,09:10:00,17:10:00,60,Present
2,Priya,HR,2026-10-13,09:20:00,16:50:00,45,Present
3,Rahul,Finance,2026-10-13,null,null,0,Absent
4,Meena,IT,2026-10-13,09:00:00,17:30:00,60,Present
1,Arun,IT,2026-10-14,09:00:00,17:00:00,60,Present
2,Priya,HR,2026-10-14,09:10:00,17:10:00,45,Present


In [0]:
print("Attendance Records:", attendance_df.count())

display(attendance_df)

Attendance Records: 12


employee_id,employee_name,department,attendance_date,clock_in,clock_out,break_minutes,attendance_status
1,Arun,IT,2026-10-12,09:00:00,17:00:00,60,Present
2,Priya,HR,2026-10-12,09:15:00,17:15:00,45,Present
3,Rahul,Finance,2026-10-12,null,null,0,Absent
4,Meena,IT,2026-10-12,09:05:00,17:05:00,60,Present
1,Arun,IT,2026-10-13,09:10:00,17:10:00,60,Present
2,Priya,HR,2026-10-13,09:20:00,16:50:00,45,Present
3,Rahul,Finance,2026-10-13,null,null,0,Absent
4,Meena,IT,2026-10-13,09:00:00,17:30:00,60,Present
1,Arun,IT,2026-10-14,09:00:00,17:00:00,60,Present
2,Priya,HR,2026-10-14,09:10:00,17:10:00,45,Present


In [0]:
attendance_df.printSchema()

root
 |-- employee_id: integer (nullable = true)
 |-- employee_name: string (nullable = true)
 |-- department: string (nullable = true)
 |-- attendance_date: date (nullable = true)
 |-- clock_in: time(6) (nullable = true)
 |-- clock_out: time(6) (nullable = true)
 |-- break_minutes: integer (nullable = true)
 |-- attendance_status: string (nullable = true)



In [0]:
tasks_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(base_path + "tasks.csv")
)

display(tasks_df)

employee_id,employee_name,department,report_date,tasks_assigned,tasks_completed
1,Arun,IT,2026-10-12,4,4
2,Priya,HR,2026-10-12,3,2
3,Rahul,Finance,2026-10-12,3,1
4,Meena,IT,2026-10-12,5,4
1,Arun,IT,2026-10-13,3,3
2,Priya,HR,2026-10-13,4,3
3,Rahul,Finance,2026-10-13,2,0
4,Meena,IT,2026-10-13,4,4
1,Arun,IT,2026-10-14,4,3
2,Priya,HR,2026-10-14,3,3


In [0]:
print("Task Records:", tasks_df.count())

tasks_df.printSchema()

Task Records: 12
root
 |-- employee_id: integer (nullable = true)
 |-- employee_name: string (nullable = true)
 |-- department: string (nullable = true)
 |-- report_date: date (nullable = true)
 |-- tasks_assigned: integer (nullable = true)
 |-- tasks_completed: integer (nullable = true)



In [0]:
from pyspark.sql.functions import *

tasks_original_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(base_path + "tasks.csv")
)

employees = tasks_original_df.select(
    "employee_id",
    "employee_name",
    "department"
).distinct()

days = spark.range(1, 6).select(
    col("id").cast("int").alias("day_no")
)

new_tasks = (
    employees
    .crossJoin(days)
    .withColumn(
        "report_date",
        date_add(current_date(), col("day_no"))
    )
    .withColumn(
        "tasks_assigned",
        (lit(5) + pmod(col("employee_id") + col("day_no"), lit(6))).cast("int")
    )
    .withColumn(
        "tasks_completed",
        pmod(
            col("employee_id") + col("day_no"),
            col("tasks_assigned") + 1
        ).cast("int")
    )
    .select(tasks_original_df.columns)
)

tasks_df = tasks_original_df.unionByName(new_tasks)

print("Original Records:", tasks_original_df.count())
print("New Records:", new_tasks.count())
print("Total Task Records:", tasks_df.count())

display(tasks_df)

Original Records: 12
New Records: 20
Total Task Records: 32


employee_id,employee_name,department,report_date,tasks_assigned,tasks_completed
1,Arun,IT,2026-10-12,4,4
2,Priya,HR,2026-10-12,3,2
3,Rahul,Finance,2026-10-12,3,1
4,Meena,IT,2026-10-12,5,4
1,Arun,IT,2026-10-13,3,3
2,Priya,HR,2026-10-13,4,3
3,Rahul,Finance,2026-10-13,2,0
4,Meena,IT,2026-10-13,4,4
1,Arun,IT,2026-10-14,4,3
2,Priya,HR,2026-10-14,3,3


In [0]:
from pyspark.sql.functions import *

attendance_clean_df = (
    attendance_df
    .dropDuplicates()
    .dropna(subset=["employee_id"])
)

display(attendance_clean_df)

print("Original Records:", attendance_df.count())
print("Clean Records:", attendance_clean_df.count())

employee_id,employee_name,department,attendance_date,clock_in,clock_out,break_minutes,attendance_status
2,Priya,HR,2026-10-14,09:10:00,17:10:00,45,Present
2,Priya,HR,2026-10-12,09:15:00,17:15:00,45,Present
3,Rahul,Finance,2026-10-12,null,null,0,Absent
1,Arun,IT,2026-10-13,09:10:00,17:10:00,60,Present
2,Priya,HR,2026-10-13,09:20:00,16:50:00,45,Present
1,Arun,IT,2026-10-14,09:00:00,17:00:00,60,Present
4,Meena,IT,2026-10-13,09:00:00,17:30:00,60,Present
4,Meena,IT,2026-10-12,09:05:00,17:05:00,60,Present
3,Rahul,Finance,2026-10-13,null,null,0,Absent
4,Meena,IT,2026-10-14,09:15:00,17:15:00,60,Present


Original Records: 12
Clean Records: 12


In [0]:
tasks_clean_df = (
    tasks_df
    .dropDuplicates()
    .dropna(subset=["employee_id"])
    .filter(col("tasks_assigned") >= 0)
    .filter(col("tasks_completed") >= 0)
)

display(tasks_clean_df)

print("Clean Task Records:", tasks_clean_df.count())

employee_id,employee_name,department,report_date,tasks_assigned,tasks_completed
2,Priya,HR,2026-10-14,3,3
3,Rahul,Finance,2026-10-14,4,1
4,Meena,IT,2026-10-13,4,4
2,Priya,HR,2026-10-13,4,3
4,Meena,IT,2026-10-14,5,4
1,Arun,IT,2026-10-12,4,4
3,Rahul,Finance,2026-10-13,2,0
1,Arun,IT,2026-10-14,4,3
4,Meena,IT,2026-10-12,5,4
2,Priya,HR,2026-10-12,3,2


Clean Task Records: 32


In [0]:
print("Attendance:", attendance_clean_df.count())
print("Tasks:", tasks_clean_df.count())

Attendance: 12
Tasks: 32


In [0]:
print("Attendance Columns:")
print(attendance_clean_df.columns)

print("Task Columns:")
print(tasks_clean_df.columns)

Attendance Columns:
['employee_id', 'employee_name', 'department', 'attendance_date', 'clock_in', 'clock_out', 'break_minutes', 'attendance_status']
Task Columns:
['employee_id', 'employee_name', 'department', 'report_date', 'tasks_assigned', 'tasks_completed']


In [0]:
display(attendance_clean_df.limit(5))

employee_id,employee_name,department,attendance_date,clock_in,clock_out,break_minutes,attendance_status
2,Priya,HR,2026-10-12,09:15:00,17:15:00,45,Present
3,Rahul,Finance,2026-10-12,null,null,0,Absent
1,Arun,IT,2026-10-13,09:10:00,17:10:00,60,Present
4,Meena,IT,2026-10-12,09:05:00,17:05:00,60,Present
1,Arun,IT,2026-10-12,09:00:00,17:00:00,60,Present


In [0]:
from pyspark.sql.functions import col

employee_combined_df = (
    attendance_clean_df.alias("a")
    .join(
        tasks_clean_df.alias("t"),
        (col("a.employee_id") == col("t.employee_id")) &
        (col("a.attendance_date") == col("t.report_date")),
        "left"
    )
    .select(
        col("a.employee_id"),
        col("a.employee_name"),
        col("a.department"),
        col("a.attendance_date"),
        col("a.clock_in"),
        col("a.clock_out"),
        col("a.break_minutes"),
        col("a.attendance_status"),
        col("t.tasks_assigned"),
        col("t.tasks_completed")
    )
)

display(employee_combined_df)

employee_id,employee_name,department,attendance_date,clock_in,clock_out,break_minutes,attendance_status,tasks_assigned,tasks_completed
2,Priya,HR,2026-10-14,09:10:00,17:10:00,45,Present,3,3
2,Priya,HR,2026-10-12,09:15:00,17:15:00,45,Present,3,2
3,Rahul,Finance,2026-10-12,null,null,0,Absent,3,1
1,Arun,IT,2026-10-13,09:10:00,17:10:00,60,Present,3,3
2,Priya,HR,2026-10-13,09:20:00,16:50:00,45,Present,4,3
1,Arun,IT,2026-10-14,09:00:00,17:00:00,60,Present,4,3
4,Meena,IT,2026-10-13,09:00:00,17:30:00,60,Present,4,4
4,Meena,IT,2026-10-12,09:05:00,17:05:00,60,Present,5,4
3,Rahul,Finance,2026-10-13,null,null,0,Absent,2,0
4,Meena,IT,2026-10-14,09:15:00,17:15:00,60,Present,5,4


In [0]:
print("Total Combined Records:", employee_combined_df.count())

employee_combined_df.printSchema()

Total Combined Records: 12
root
 |-- employee_id: integer (nullable = true)
 |-- employee_name: string (nullable = true)
 |-- department: string (nullable = true)
 |-- attendance_date: date (nullable = true)
 |-- clock_in: time(6) (nullable = true)
 |-- clock_out: time(6) (nullable = true)
 |-- break_minutes: integer (nullable = true)
 |-- attendance_status: string (nullable = true)
 |-- tasks_assigned: integer (nullable = true)
 |-- tasks_completed: integer (nullable = true)



In [0]:
employee_combined_df = employee_combined_df.withColumn(
    "task_completion_percentage",
    when(
        col("tasks_assigned") > 0,
        round(
            col("tasks_completed") / col("tasks_assigned") * 100,
            2
        )
    ).otherwise(0)
)

display(employee_combined_df)

employee_id,employee_name,department,attendance_date,clock_in,clock_out,break_minutes,attendance_status,tasks_assigned,tasks_completed,task_completion_percentage
2,Priya,HR,2026-10-14,09:10:00,17:10:00,45,Present,3,3,100.0
2,Priya,HR,2026-10-12,09:15:00,17:15:00,45,Present,3,2,66.67
3,Rahul,Finance,2026-10-12,null,null,0,Absent,3,1,33.33
1,Arun,IT,2026-10-13,09:10:00,17:10:00,60,Present,3,3,100.0
2,Priya,HR,2026-10-13,09:20:00,16:50:00,45,Present,4,3,75.0
1,Arun,IT,2026-10-14,09:00:00,17:00:00,60,Present,4,3,75.0
4,Meena,IT,2026-10-13,09:00:00,17:30:00,60,Present,4,4,100.0
4,Meena,IT,2026-10-12,09:05:00,17:05:00,60,Present,5,4,80.0
3,Rahul,Finance,2026-10-13,null,null,0,Absent,2,0,0.0
4,Meena,IT,2026-10-14,09:15:00,17:15:00,60,Present,5,4,80.0


In [0]:

from pyspark.sql.functions import *

employee_kpi_df = (
    employee_combined_df
    .groupBy("employee_id", "employee_name", "department")
    .agg(
        count("attendance_date").alias("total_attendance_days"),
        sum(when(col("attendance_status") == "Present", 1).otherwise(0)).alias("present_days"),
        sum(when(col("attendance_status") == "Absent", 1).otherwise(0)).alias("absent_days"),
        sum("tasks_assigned").alias("total_tasks_assigned"),
        sum("tasks_completed").alias("total_tasks_completed")
    )
    .withColumn(
        "attendance_percentage",
        round(col("present_days") / col("total_attendance_days") * 100, 2)
    )
    .withColumn(
        "task_completion_percentage",
        when(
            col("total_tasks_assigned") > 0,
            round(col("total_tasks_completed") / col("total_tasks_assigned") * 100, 2)
        ).otherwise(0)
    )
)

display(employee_kpi_df)

employee_id,employee_name,department,total_attendance_days,present_days,absent_days,total_tasks_assigned,total_tasks_completed,attendance_percentage,task_completion_percentage
2,Priya,HR,3,3,0,10,8,100.0,80.0
3,Rahul,Finance,3,0,3,9,2,0.0,22.22
1,Arun,IT,3,3,0,11,10,100.0,90.91
4,Meena,IT,3,3,0,14,12,100.0,85.71


In [0]:
department_kpi_df = (
    employee_kpi_df
    .groupBy("department")
    .agg(
        countDistinct("employee_id").alias("total_employees"),
        sum("total_attendance_days").alias("total_attendance_days"),
        sum("present_days").alias("total_present_days"),
        sum("absent_days").alias("total_absent_days"),
        sum("total_tasks_assigned").alias("total_tasks_assigned"),
        sum("total_tasks_completed").alias("total_tasks_completed")
    )
    .withColumn(
        "attendance_percentage",
        when(
            col("total_attendance_days") > 0,
            round(col("total_present_days") / col("total_attendance_days") * 100, 2)
        ).otherwise(0)
    )
    .withColumn(
        "task_completion_percentage",
        when(
            col("total_tasks_assigned") > 0,
            round(col("total_tasks_completed") / col("total_tasks_assigned") * 100, 2)
        ).otherwise(0)
    )
)

display(department_kpi_df)

department,total_employees,total_attendance_days,total_present_days,total_absent_days,total_tasks_assigned,total_tasks_completed,attendance_percentage,task_completion_percentage
HR,1,3,3,0,10,8,100.0,80.0
Finance,1,3,0,3,9,2,0.0,22.22
IT,2,6,6,0,25,22,100.0,88.0


In [0]:

employee_kpi_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.employee.employee_kpi")

In [0]:
%sql
SELECT *
FROM workspace.employee.employee_kpi;

employee_id,employee_name,department,total_attendance_days,present_days,absent_days,total_tasks_assigned,total_tasks_completed,attendance_percentage,task_completion_percentage
4,Meena,IT,3,3,0,14,12,100.0,85.71
1,Arun,IT,3,3,0,11,10,100.0,90.91
2,Priya,HR,3,3,0,10,8,100.0,80.0
3,Rahul,Finance,3,0,3,9,2,0.0,22.22


In [0]:

department_kpi_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.employee.department_kpi")

In [0]:
%sql
SELECT *
FROM workspace.employee.department_kpi;

department,total_employees,total_attendance_days,total_present_days,total_absent_days,total_tasks_assigned,total_tasks_completed,attendance_percentage,task_completion_percentage
IT,2,6,6,0,25,22,100.0,88.0
HR,1,3,3,0,10,8,100.0,80.0
Finance,1,3,0,3,9,2,0.0,22.22


In [0]:
%python
department_kpi_df.coalesce(1).write \
    .format("csv") \
    .mode("overwrite") \
    .option("header", True) \
    .save(base_path + "department_kpi_export")

In [0]:
%python
display(dbutils.fs.ls(base_path + "department_kpi_export"))

path,name,size,modificationTime
dbfs:/Volumes/workspace/employee/employee_assessment_files/department_kpi_export/_SUCCESS,_SUCCESS,0,1794398611000
dbfs:/Volumes/workspace/employee/employee_assessment_files/department_kpi_export/_committed_489909935018618098,_committed_489909935018618098,113,1794398611000
dbfs:/Volumes/workspace/employee/employee_assessment_files/department_kpi_export/_started_489909935018618098,_started_489909935018618098,0,1794398610000
dbfs:/Volumes/workspace/employee/employee_assessment_files/department_kpi_export/part-00000-tid-489909935018618098-f8b32d43-ad81-42a9-bf57-1b5d0989a4b7-1923-1-c000.csv,part-00000-tid-489909935018618098-f8b32d43-ad81-42a9-bf57-1b5d0989a4b7-1923-1-c000.csv,263,1794398611000


In [0]:
%sql
SHOW TABLES IN workspace.employee;

database,tableName,isTemporary
employee,department_kpi,false
employee,employee_kpi,false


In [0]:

from pyspark.sql.functions import *

attendance_employee_df = (
    attendance_clean_df
    .groupBy("employee_id", "employee_name", "department")
    .agg(
        count("attendance_date").alias("total_attendance_days"),
        sum(when(lower(trim(col("attendance_status"))) == "present", 1).otherwise(0)).alias("present_days"),
        sum(when(lower(trim(col("attendance_status"))) == "absent", 1).otherwise(0)).alias("absent_days")
    )
)

task_employee_df = (
    tasks_clean_df
    .groupBy("employee_id")
    .agg(
        sum("tasks_assigned").alias("total_tasks_assigned"),
        sum("tasks_completed").alias("total_tasks_completed")
    )
)

employee_kpi_df = (
    attendance_employee_df
    .join(task_employee_df, "employee_id", "left")
    .fillna(0, subset=["total_tasks_assigned", "total_tasks_completed"])
    .withColumn(
        "attendance_percentage",
        round(col("present_days") / col("total_attendance_days") * 100, 2)
    )
    .withColumn(
        "task_completion_percentage",
        when(
            col("total_tasks_assigned") > 0,
            round(col("total_tasks_completed") / col("total_tasks_assigned") * 100, 2)
        ).otherwise(0)
    )
)

display(employee_kpi_df)

employee_id,employee_name,department,total_attendance_days,present_days,absent_days,total_tasks_assigned,total_tasks_completed,attendance_percentage,task_completion_percentage
4,Meena,IT,3,3,0,50,17,100.0,34.0
1,Arun,IT,3,3,0,50,24,100.0,48.0
2,Priya,HR,3,3,0,48,20,100.0,41.67
3,Rahul,Finance,3,0,3,46,11,0.0,23.91


In [0]:

department_kpi_df = (
    employee_kpi_df
    .groupBy("department")
    .agg(
        countDistinct("employee_id").alias("total_employees"),
        sum("total_attendance_days").alias("total_attendance_days"),
        sum("present_days").alias("total_present_days"),
        sum("absent_days").alias("total_absent_days"),
        sum("total_tasks_assigned").alias("total_tasks_assigned"),
        sum("total_tasks_completed").alias("total_tasks_completed")
    )
    .withColumn(
        "attendance_percentage",
        round(col("total_present_days") / col("total_attendance_days") * 100, 2)
    )
    .withColumn(
        "task_completion_percentage",
        when(
            col("total_tasks_assigned") > 0,
            round(col("total_tasks_completed") / col("total_tasks_assigned") * 100, 2)
        ).otherwise(0)
    )
)

display(department_kpi_df)

department,total_employees,total_attendance_days,total_present_days,total_absent_days,total_tasks_assigned,total_tasks_completed,attendance_percentage,task_completion_percentage
IT,2,6,6,0,100,41,100.0,41.0
HR,1,3,3,0,48,20,100.0,41.67
Finance,1,3,0,3,46,11,0.0,23.91


In [0]:

employee_kpi_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.employee.employee_kpi")

department_kpi_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.employee.department_kpi")

print("Both KPI tables updated successfully!")

Both KPI tables updated successfully!


In [0]:
employee_kpi_df.coalesce(1).write \
    .mode("overwrite") \
    .option("header", True) \
    .csv(base_path + "employee_kpi_export")

department_kpi_df.coalesce(1).write \
    .mode("overwrite") \
    .option("header", True) \
    .csv(base_path + "department_kpi_export")

print("All corrected KPI exports completed!")

All corrected KPI exports completed!
